**Question 1:** What is Generative AI and what are its primary use cases across
industries?

 - Generative AI is a type of Artificial Intelligence that can create new content such as text, images, audio, video, or code by learning patterns from existing data. It uses models like GANs, VAEs, and Transformer-based models (e.g., GPT).

**Primary Use Cases Across Industries:**

**1.** Healthcare

 - Drug discovery

 - Medical report generation

**2.** Finance

 - Fraud detection support

 - Automated financial report generation

**3.** Media & Entertainment

 - Image, music, and video generation

 - Script or content writing

**4.** Software Development

 - Code generation and debugging

**5.** Marketing

 - Ad copy generation

 - Personalized content creation

**6.** Education

 - Automated tutoring

 - Study material generation

 ---

**Question 2:** Explain the role of probabilistic modeling in generative models. How do these models differ from discriminative models?

 - **Role of Probabilistic Modeling in Generative Models**

Probabilistic modeling is used in generative models to learn the probability distribution of the data.
The model tries to estimate P(X, Y) (joint probability of input and output).

This allows the model to:

 - Generate new data samples similar to the training data

 - Handle uncertainty in data

 - Model how the data is actually distributed

**Example:** Models like Gaussian Mixture Models, Naive Bayes, GANs, and VAEs use probabilistic concepts to generate new data.

**Difference Between Generative and Discriminative Models**

| Aspect              | Generative Models                             | Discriminative Models                     |
| ------------------- | --------------------------------------------- | ----------------------------------------- |
| Probability Learned | Learn **P(X, Y)**                             | Learn **P(Y|X)**                          |
| Goal                | Model data distribution and generate new data | Predict label from input                  |
| Capability          | Can generate new samples                      | Cannot generate new data                  |
| Examples            | Naive Bayes, GAN, VAE                         | Logistic Regression, SVM, Neural Networks |

---

**Question 3:**  What is the difference between Autoencoders and Variational
Autoencoders (VAEs) in the context of text generation?

 - **Difference Between Autoencoders and Variational Autoencoders (VAEs)**

 | Aspect          | Autoencoder                                   | Variational Autoencoder (VAE)                                            |
| --------------- | --------------------------------------------- | ------------------------------------------------------------------------ |
| Concept         | Learns to **compress and reconstruct data**   | Learns a **probability distribution of data**                            |
| Latent Space    | Fixed encoded vector                          | Latent variables follow a **probability distribution (mean & variance)** |
| Data Generation | Mainly used for **reconstruction**            | Can **generate new data samples**                                        |
| Output          | Reconstructed input                           | New samples generated from latent distribution                           |
| Use in Text     | Used for **text representation or denoising** | Used for **text generation and variation**                               |

---

**Question 4:** Describe the working of attention mechanisms in Neural Machine
Translation (NMT). Why are they critical?

 - In Neural Machine Translation, the attention mechanism helps the model focus on the most relevant words in the source sentence while generating each word in the translated sentence.

**Working:**

 - The encoder converts the input sentence into a set of hidden states (one for each word).

 - During translation, the decoder generates one word at a time.

 - For each output word, the attention mechanism calculates weights for all encoder hidden states.

- The model gives higher weight to important words and combines them to create a context vector.

 - The decoder uses this context vector to produce the next translated word.

**Why Attention is Critical:**

 - Helps the model focus on relevant words instead of the whole sentence equally.

 - Improves translation accuracy, especially for long sentences.

 - Solves the information bottleneck problem of traditional encoder–decoder models.

 - Enables better word alignment between source and target sentences.

 ---

**Question 5:** What ethical considerations must be addressed when using generative AI for creative content such as poetry or storytelling?

 - **Ethical Considerations in Generative AI for Creative Content**

**1.** **Copyright and Ownership**

AI-generated content may be similar to existing works, raising questions about who owns the content and whether original creators’ rights are violated.

**2.** **Plagiarism**

The model may unintentionally replicate parts of existing poems or stories from training data.

**3.** **Bias and Fairness**

If training data contains bias, the generated content may produce stereotypes or unfair representations.

**4.** **Misinformation**

AI may generate false or misleading stories that appear realistic.

**5.** **Transparency**

It should be clearly disclosed when content is generated by AI instead of a human.

**6.** **Misuse of Creativity**

AI could be used to impersonate authors or create harmful or offensive content.

---

**Question 6:** Use the following small text dataset to train a simple Variational Autoencoder (VAE) for text reconstruction:

["The sky is blue", "The sun is bright", "The grass is green",
"The night is dark", "The stars are shining"]

1. Preprocess the data (tokenize and pad the sequences).
2. Build a basic VAE model for text reconstruction.
3. Train the model and show how it reconstructs or generates similar sentences.


In [ ]:
# Import Libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Lambda, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras.losses import sparse_categorical_crossentropy

# Dataset
sentences = [
    "The sky is blue",
    "The sun is bright",
    "The grass is green",
    "The night is dark",
    "The stars are shining"
]

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)

# Padding
max_len = max(len(seq) for seq in sequences)
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

vocab_size = len(tokenizer.word_index) + 1

# Model parameters
embedding_dim = 8
latent_dim = 2

# Encoder
inputs = Input(shape=(max_len,))
x = Embedding(vocab_size, embedding_dim)(inputs)
x = Flatten()(x)

z_mean = Dense(latent_dim)(x)
z_log_var = Dense(latent_dim)(x)

# Sampling (Reparameterization Trick)
def sampling(args):
    z_mean, z_log_var = args
    epsilon = tf.random.normal(shape=(tf.shape(z_mean)[0], latent_dim))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

z = Lambda(sampling)([z_mean, z_log_var])

# Decoder
decoder_dense = Dense(16, activation='relu')(z)
decoder_output = Dense(max_len * vocab_size, activation='softmax')(decoder_dense)
outputs = Reshape((max_len, vocab_size))(decoder_output)

# VAE Model
vae = Model(inputs, outputs)

# Loss Function
reconstruction_loss = sparse_categorical_crossentropy(padded_sequences, outputs)
reconstruction_loss = tf.reduce_mean(reconstruction_loss)

kl_loss = -0.5 * tf.reduce_mean(
    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
)

vae_loss = reconstruction_loss + kl_loss
vae.add_loss(vae_loss)

vae.compile(optimizer='adam')

# Train Model
vae.fit(padded_sequences, epochs=200, batch_size=2, verbose=0)

# Reconstruction
pred = vae.predict(padded_sequences)
pred_words = np.argmax(pred, axis=-1)

# Convert back to text
reverse_word_index = {v:k for k,v in tokenizer.word_index.items()}

print("Reconstructed Sentences:\n")

for seq in pred_words:
    sentence = []
    for idx in seq:
        word = reverse_word_index.get(idx, "")
        if word != "":
            sentence.append(word)
    print(" ".join(sentence))

**Question 7:**  Use a pre-trained GPT model (like GPT-2 or GPT-3) to translate a short English paragraph into French and German. Provide the original and translated text.

In [ ]:
# Install transformers
!pip install transformers

# Import libraries
from transformers import pipeline

# Load GPT-2 text generation pipeline
generator = pipeline("text-generation", model="gpt2")

# Original English paragraph
text = "Artificial Intelligence is transforming the world. It helps machines learn from data and perform tasks that normally require human intelligence."

# Prompt for French translation
prompt_fr = f"Translate the following English text to French:\n{text}\nFrench:"
result_fr = generator(prompt_fr, max_length=120, num_return_sequences=1)

# Prompt for German translation
prompt_de = f"Translate the following English text to German:\n{text}\nGerman:"
result_de = generator(prompt_de, max_length=120, num_return_sequences=1)

# Print results
print("Original English Text:\n", text)

print("\nFrench Translation:\n")
print(result_fr[0]['generated_text'])

print("\nGerman Translation:\n")
print(result_de[0]['generated_text'])

**Question 8:** Implement a simple attention-based encoder-decoder model for
English-to-Spanish translation using Tensorflow or PyTorch.


In [ ]:
# Install TensorFlow (if not installed)
!pip install tensorflow

import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ---------------------------
# 1. Small English-Spanish Dataset
# ---------------------------
eng_sentences = [
    "hello",
    "how are you",
    "i am fine",
    "good morning",
    "thank you"
]

spa_sentences = [
    "hola",
    "como estas",
    "estoy bien",
    "buenos dias",
    "gracias"
]

# ---------------------------
# 2. Tokenization
# ---------------------------
eng_tokenizer = Tokenizer()
spa_tokenizer = Tokenizer()

eng_tokenizer.fit_on_texts(eng_sentences)
spa_tokenizer.fit_on_texts(spa_sentences)

eng_seq = eng_tokenizer.texts_to_sequences(eng_sentences)
spa_seq = spa_tokenizer.texts_to_sequences(spa_sentences)

# ---------------------------
# 3. Padding
# ---------------------------
max_eng_len = max(len(s) for s in eng_seq)
max_spa_len = max(len(s) for s in spa_seq)

eng_pad = pad_sequences(eng_seq, maxlen=max_eng_len, padding='post')
spa_pad = pad_sequences(spa_seq, maxlen=max_spa_len, padding='post')

eng_vocab_size = len(eng_tokenizer.word_index) + 1
spa_vocab_size = len(spa_tokenizer.word_index) + 1

# ---------------------------
# 4. Model Parameters
# ---------------------------
embedding_dim = 64
units = 128

# ---------------------------
# 5. Encoder
# ---------------------------
encoder_inputs = tf.keras.Input(shape=(max_eng_len,))
encoder_embedding = tf.keras.layers.Embedding(eng_vocab_size, embedding_dim)(encoder_inputs)

encoder_outputs, state_h, state_c = tf.keras.layers.LSTM(
    units,
    return_sequences=True,
    return_state=True
)(encoder_embedding)

# ---------------------------
# 6. Decoder
# ---------------------------
decoder_inputs = tf.keras.Input(shape=(max_spa_len,))
decoder_embedding = tf.keras.layers.Embedding(spa_vocab_size, embedding_dim)(decoder_inputs)

decoder_lstm = tf.keras.layers.LSTM(
    units,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=[state_h, state_c]
)

# ---------------------------
# 7. Attention Mechanism
# ---------------------------
attention = tf.keras.layers.Attention()
context_vector = attention([decoder_outputs, encoder_outputs])

# Combine decoder output + context
concat = tf.keras.layers.Concatenate(axis=-1)([decoder_outputs, context_vector])

# Output layer
outputs = tf.keras.layers.Dense(spa_vocab_size, activation='softmax')(concat)

# ---------------------------
# 8. Build Model
# ---------------------------
model = tf.keras.Model([encoder_inputs, decoder_inputs], outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ---------------------------
# 9. Train Model
# ---------------------------
model.fit(
    [eng_pad, spa_pad],
    np.expand_dims(spa_pad, -1),
    epochs=200,
    verbose=0
)

print("Training Completed")

# ---------------------------
# 10. Test Translation
# ---------------------------
test_sentence = ["hello"]

seq = eng_tokenizer.texts_to_sequences(test_sentence)
seq = pad_sequences(seq, maxlen=max_eng_len, padding='post')

prediction = model.predict([seq, np.zeros((1, max_spa_len))])

pred_words = np.argmax(prediction, axis=-1)

reverse_spa_index = {v:k for k,v in spa_tokenizer.word_index.items()}

translated_sentence = []
for idx in pred_words[0]:
    word = reverse_spa_index.get(idx, "")
    if word != "":
        translated_sentence.append(word)

print("English:", test_sentence[0])
print("Spanish:", " ".join(translated_sentence))

**Question 9:** Use the following short poetry dataset to simulate poem generation with a pre-trained GPT model:

["Roses are red, violets are blue,",
"Sugar is sweet, and so are you.",
"The moon glows bright in silent skies,",
"A bird sings where the soft wind sighs."]

Using this dataset as a reference for poetic structure and language, generate a new 2-4 line poem using a pre-trained GPT model (such as GPT-2). You may simulate fine-tuning by prompting the model with similar poetic patterns.



In [ ]:
# Install transformers
!pip install transformers

from transformers import pipeline

# -----------------------------
# Poetry Dataset (reference)
# -----------------------------
dataset = [
"Roses are red, violets are blue,",
"Sugar is sweet, and so are you.",
"The moon glows bright in silent skies,",
"A bird sings where the soft wind sighs."
]

# -----------------------------
# Load Pre-trained GPT-2 Model
# -----------------------------
generator = pipeline("text-generation", model="gpt2")

# -----------------------------
# Prompt (Simulating Fine-tuning)
# -----------------------------
prompt = """
Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.

Write a similar 4 line poem:
"""

# -----------------------------
# Generate Poem
# -----------------------------
result = generator(prompt, max_length=80, num_return_sequences=1)

# -----------------------------
# Display Results
# -----------------------------
print("Poetry Dataset:")
for line in dataset:
    print(line)

print("\nPrompt Used:")
print(prompt)

print("\nGenerated Poem:")
print(result[0]['generated_text'])

**Question 10:** Imagine you are building a creative writing assistant for a publishing company. The assistant should generate story plots and character descriptions using Generative AI. Describe how you would design the system, including model selection, training data, bias mitigation, and evaluation methods. Explain the real-world challenges you might face.


In [ ]:
# Install required library
!pip install transformers

from transformers import pipeline

# -----------------------------
# Load Pre-trained GPT Model
# -----------------------------
generator = pipeline("text-generation", model="gpt2")

# -----------------------------
# User Inputs
# -----------------------------
genre = "mystery"
theme = "lost treasure"
setting = "an ancient island"

# -----------------------------
# Prompt for Story Plot
# -----------------------------
plot_prompt = f"""
Write a short story plot for a {genre} story.
Theme: {theme}
Setting: {setting}

Story Plot:
"""

# Generate Story Plot
plot = generator(plot_prompt, max_length=120, num_return_sequences=1)

# -----------------------------
# Prompt for Character Description
# -----------------------------
character_prompt = f"""
Create a detailed character description for the main character of a {genre} story.
Theme: {theme}
Setting: {setting}

Character Description:
"""

character = generator(character_prompt, max_length=120, num_return_sequences=1)

# -----------------------------
# Output
# -----------------------------
print("Genre:", genre)
print("Theme:", theme)
print("Setting:", setting)

print("\nGenerated Story Plot:\n")
print(plot[0]['generated_text'])

print("\nGenerated Character Description:\n")
print(character[0]['generated_text'])